# Ch 01 — 선형 회귀 (최소제곱법)

원본: `deep_class/01_Linear_Regression.py`

다루는 내용:
1. 데이터 정의 및 시각화
2. 최소제곱법 — 책 스타일 (`for` 루프) → NumPy 벡터화
3. `np.polyfit` 으로 검증
4. `sklearn.linear_model.LinearRegression` 으로 재확인
5. (보너스) Keras 3 — 1뉴런 회귀 모델로 같은 직선 찾기

## 0. 환경 확인 / 시드

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import matplotlib.pyplot as plt
import keras
import tensorflow as tf

keras.utils.set_random_seed(42)

print("NumPy     :", np.__version__)
print("TensorFlow:", tf.__version__)
print("Keras     :", keras.__version__)

## 1. 데이터 — 공부시간 vs 점수

| x (공부시간) | 2 | 4 | 6 | 8 |
| ---------- | - | - | - | - |
| y (점수)    | 81 | 93 | 91 | 97 |

In [ ]:
x = np.array([2, 4, 6, 8], dtype=float)
y = np.array([81, 93, 91, 97], dtype=float)

mx, my = x.mean(), y.mean()
print(f"x의 평균값: {mx}")
print(f"y의 평균값: {my}")

In [ ]:
plt.figure(figsize=(5, 4))
plt.scatter(x, y, s=80, label="data")
plt.axhline(my, color="gray", lw=0.5, ls="--")
plt.axvline(mx, color="gray", lw=0.5, ls="--")
plt.scatter([mx], [my], color="red", s=120, marker="x", label=f"mean ({mx}, {my})")
plt.xlabel("x (공부시간)")
plt.ylabel("y (점수)")
plt.title("Ch01 — Raw data")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 2. 최소제곱법

$$a = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sum (x_i - \bar{x})^2}, \quad b = \bar{y} - a\bar{x}$$

### 2-1. 책 스타일 (`for` 루프)

In [ ]:
divisor = sum([(mx - i) ** 2 for i in x])

def top(x, mx, y, my):
    d = 0
    for i in range(len(x)):
        d += (x[i] - mx) * (y[i] - my)
    return d

dividend = top(x, mx, y, my)

a = dividend / divisor
b = my - mx * a

print(f"분모 = {divisor}")
print(f"분자 = {dividend}")
print(f"기울기 a = {a}")
print(f"y절편  b = {b}")

### 2-2. NumPy 벡터화

같은 식을 한 줄로. 데이터가 커질수록 차이가 큼.

In [ ]:
a_vec = ((x - mx) * (y - my)).sum() / ((x - mx) ** 2).sum()
b_vec = my - a_vec * mx

print(f"기울기 a = {a_vec}")
print(f"y절편  b = {b_vec}")
assert np.isclose(a, a_vec) and np.isclose(b, b_vec)

## 3. `np.polyfit` 으로 검증

1차 다항식 = 직선. NumPy가 같은 답을 주는지.

In [ ]:
a_np, b_np = np.polyfit(x, y, deg=1)
print(f"np.polyfit  → a = {a_np:.6f}, b = {b_np:.6f}")
print(f"수동 계산   → a = {a:.6f}, b = {b:.6f}")

## 4. scikit-learn `LinearRegression`

본격적으로 ML 라이브러리를 써보기. 입력은 2D shape `(N, 1)` 필요.

In [ ]:
from sklearn.linear_model import LinearRegression

X = x.reshape(-1, 1)
model = LinearRegression().fit(X, y)
print(f"sklearn → a = {model.coef_[0]:.6f}, b = {model.intercept_:.6f}")

## 5. 결과 직선 시각화

In [ ]:
xs = np.linspace(x.min() - 1, x.max() + 1, 50)
ys = a * xs + b

plt.figure(figsize=(5, 4))
plt.scatter(x, y, s=80, label="data")
plt.plot(xs, ys, color="crimson", label=f"y = {a:.2f}·x + {b:.2f}")
plt.scatter([mx], [my], color="red", s=120, marker="x", label="mean point")
plt.xlabel("x (공부시간)")
plt.ylabel("y (점수)")
plt.title("Ch01 — 최소제곱 직선")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 6. (보너스) Keras 3 — 1뉴런 회귀 모델

`Dense(1)` 한 개로 `y = W·x + b` 를 표현하고 경사하강법으로 학습. 다음 장(경사하강) 으로 가는 다리.

- Keras 3 권장 패턴: `Sequential([Input(shape=...), Dense(...)])`
- 데이터가 4개뿐이라 학습률·에폭을 충분히 크게 잡아야 수렴함.

In [ ]:
from keras import Input, Sequential
from keras.layers import Dense
from keras.optimizers import SGD

model = Sequential([
    Input(shape=(1,)),
    Dense(1),
])
model.compile(optimizer=SGD(learning_rate=0.01), loss="mse")

history = model.fit(X, y, epochs=2000, verbose=0)

W, b_keras = model.layers[0].get_weights()
print(f"Keras 3 → a = {float(W[0, 0]):.6f}, b = {float(b_keras[0]):.6f}")
print(f"수동    → a = {a:.6f}, b = {b:.6f}")

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(history.history["loss"])
plt.xlabel("epoch")
plt.ylabel("MSE")
plt.title("Keras 1뉴런 회귀 — 학습 손실")
plt.yscale("log")
plt.grid(alpha=0.3)
plt.show()

## 마무리 — 체크리스트

- [ ] 최소제곱 공식의 분자/분모를 손으로 계산해 본다
- [ ] 책의 `for` 루프 코드와 NumPy 벡터화 코드가 같은 답을 내는지 확인
- [ ] `np.polyfit`, `sklearn.LinearRegression`, Keras 1뉴런 모델 — 네 방법이 모두 같은 직선에 수렴
- [ ] Keras 모델이 학습률에 얼마나 민감한지 (`learning_rate`를 0.001 / 0.05 로 바꿔보기)

**다음 장 미리보기:** 같은 문제를 *공식 없이* 손실의 기울기를 따라 내려가는 **경사하강법**으로 푼다.